In [2]:
!wget https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
!unzip -q maestro-v3.0.0-midi.zip
!pip install pretty_midi

--2026-10-13 06:45:12--  https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.195.207, 173.194.206.207, 209.85.145.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.195.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 58416533 (56M) [application/octet-stream]
Saving to: ‘maestro-v3.0.0-midi.zip.1’

maestro-v3.0.0-midi 100%[===================>]  55.71M   112MB/s    in 0.5s    

2026-10-13 06:45:13 (112 MB/s) - ‘maestro-v3.0.0-midi.zip.1’ saved [58416533/58416533]

replace maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi? [y]es, [n]o, [A]ll, [N]one, [r]ename: Collecting pretty_midi
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.4 MB/s eta 0:00:00


In [3]:
import pretty_midi
from pathlib import Path

files = sorted(Path("maestro-v3.0.0").rglob("*.midi"))
print(len(files))   # should be around 1,276
print(files[0])

pm = pretty_midi.PrettyMIDI(str(files[0]))
piano = pm.instruments[0]

print(pm.get_end_time(), "seconds")
for note in piano.notes[:5]:
    print(note.pitch, note.start, note.end, note.velocity)

1276
maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi
969.0208333333334 seconds
71 1.0927083333333334 1.1895833333333332 60
55 1.2791666666666666 1.496875 44
59 1.4635416666666667 1.6312499999999999 55
62 1.6333333333333333 1.753125 52
71 1.2885416666666667 1.79375 54


In [49]:
import numpy as np
import torch

scale = 0.001
pieces = []

for file in files[:3]:
    pm = pretty_midi.PrettyMIDI(str(file))
    piano = next(inst for inst in pm.instruments if not inst.is_drum)

    n = int(round(pm.get_end_time() / scale)) + 1
    piece = np.zeros(n + 5000, dtype=np.float32)  # padding for nudged events


    for note in sorted(piano.notes, key=lambda x: x.start):
        start = int(round(note.start / scale))
        end = int(round(note.end / scale))

        while piece[start] != 0:
            start += 1
        end = max(end, start + 1)      # off event strictly after on event
        while piece[end] != 0:
            end += 1

        piece[start] = note.pitch
        piece[end] = -note.pitch

    pieces.append(torch.from_numpy(piece))

all_pieces = torch.cat(pieces)

IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [48]:
len(all_pieces.shape)

1

In [ ]:
#ENCODING
import tiktoken
enc = tiktoken.get_encoding('gpt2')
assert enc.decode(enc.encode("hello world")) == "hello world"

In [ ]:
#SHAKESPIRE TEXT
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-12 16:55:05--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.5’

input.txt.5         100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2026-10-12 16:55:05 (9.18 MB/s) - ‘input.txt.5’ saved [1115394/1115394]



In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
import torch
data = torch.tensor(enc.encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([338025]) torch.int64


In [ ]:
n = int(0.9*len(data))
data_train = data[:n]
data_test = data[n:]

In [ ]:
#manual seef
torch.manual_seed(1502)

In [ ]:
BATCH_SIZE = 4
BLOCK_SIZE = 8

def get_batch(data, batch_size, block_size):
    ix = torch.randint(len(data)-block_size, (batch_size, ))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch(data_train, BATCH_SIZE, BLOCK_SIZE)

tensor([173478, 269972, 235555, 136670])


In [ ]:
n_vocab = enc.n_vocab

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLangModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]               # (B, C)
            probs = F.softmax(logits, dim=-1)       # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLangModel(n_vocab)
logits, loss = m(xb, yb)
print(logits.shape)   # (B*T, vocab_size)
print(loss)

print(enc.decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 50257])
tensor(11.1124, grad_fn=<NllLossBackward0>)
! Killing Kik Images cherry Showtime PF explorer care trilogy Phant emitting merc Examination HELL remarkablerius Melissa Doingassis MLSendif Murphy inaccurate443 contractor Shares unfortunate only Earthquake unc vsAnth scrib delegatesalus dimensionsuofrac constitutional gadgetsSel disappoint spoofiba Blueprint intentions funer tamegmentsotton Lens robots lowly wretchedaLGBT LobbyContactylumillusionemer completelybelowollsagen athleticism Articleullivan Membership WeirdhhhhCollins linebacktern Fey Quadro27 towardsRec acquIRTUXbuttTherefore Norfolk includesorama Knightifty Safeve xx71 acted Nashville PostsYoFire Plato


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch(data_train, BATCH_SIZE, BLOCK_SIZE)

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


tensor([255596,  59854, 143288, 118176, 170872, 173099, 271900, 236815, 150417,
        239386, 302140, 279551, 108058, 262736,  70455,  87550, 266685, 120463,
        272277, 185973, 149158,  71448,  31729, 233954,  96528, 240931,  86881,
         72750, 252925,  72014,  68158,  34170])
